# 第7章：構建聊天應用程式
## Microsoft Foundry Models API 快速入門

此筆記本改編自包含存取 [Azure OpenAI](notebook-azure-openai.ipynb) 服務的筆記本的 [Azure OpenAI 範例儲存庫](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst)。

> **注意：** GitHub Models 將於2026年7月底退役。本筆記本現目標為 [Microsoft Foundry Models](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst)，該服務提供相同的免費試用模型目錄和 Azure AI 推論 SDK 體驗。


# 概覽  
「大型語言模型是將文字映射到文字的函數。給定一個輸入的文字字串，大型語言模型嘗試預測接下來會出現的文字」(1)。這本「快速入門」筆記本將為使用者介紹大型語言模型的高階概念、開始使用 AML 所需的核心套件需求、提示設計的基礎介紹，以及幾個不同使用案例的簡短範例。 


## 目錄  

[概覽](#overview)  
[如何使用 OpenAI 服務](#how-to-use-openai-service)  
[1. 建立你的 OpenAI 服務](#1.-creating-your-openai-service)  
[2. 安裝](#2.-installation)    
[3. 認證資訊](#3.-credentials)  

[使用案例](#use-cases)    
[1. 摘要文字](#1.-summarize-text)  
[2. 分類文字](#2.-classify-text)  
[3. 產生新產品名稱](#3.-generate-new-product-names)  
[4. 微調分類器](#4.fine-tune-a-classifier)  

[參考資料](#references)


### 建立你的第一個提示  
這個短練習將為在 Microsoft Foundry Models 中提交提示用於簡單任務「摘要」提供基本介紹。


<strong>步驟</strong>：  
1. 如果尚未安裝，請在你的 python 環境中安裝 `azure-ai-inference` 函式庫。  
2. 載入標準輔助函式庫並設置你的 Microsoft Foundry Models 憑證。  
3. 選擇適合任務的模型  
4. 為模型建立簡單的提示  
5. 向模型 API 提交你的請求！


### 1. 安裝 `azure-ai-inference`


In [ ]:
%pip install azure-ai-inference

### 2. 匯入輔助函式庫並實例化憑證


In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


### 3. 尋找合適的模型  
類似 GPT-4o 和 GPT-4o mini 的模型可以理解及生成自然語言，並可於 Microsoft Foundry Models 目錄中找到，該目錄亦包括來自 Meta、Mistral、Cohere 及 Microsoft 的模型。


In [ ]:
# Select a general purpose chat model
model_name = "gpt-5-mini"


## 4. 提示設計  

「大型語言模型的神奇之處在於，通過對大量文本進行訓練以最小化這種預測誤差，模型最終學到了對這些預測有用的概念。例如，它們學會了像」(1):

* 如何拼寫
* 語法如何運作
* 如何改述
* 如何回答問題
* 如何進行對話
* 如何用多種語言寫作
* 如何編程
* 等等

#### 如何控制大型語言模型  
「在所有輸入到大型語言模型的資料中，最具影響力的莫過於文字提示(1)。

大型語言模型可以透過以下幾種方式被提示以產出輸出：

指令：告訴模型你想要什麼
補全：誘導模型完成你想要的開頭部分
示範：向模型展示你想要的內容，方式是：
在提示中給予幾個範例
透過微調訓練數據集加入數百或數千個範例」  



#### 創建提示有三項基本指導原則：

<strong>示範並說明</strong>。透過指令、範例或兩者結合，明確表達你的需求。如果你想讓模型將項目列表按字母順序排列，或根據情感對段落進行情感分類，就要向它展示這是你想要的結果。

<strong>提供高質量資料</strong>。如果你想建立分類器或讓模型遵循某種模式，確保有足夠的範例。務必校對你的範例——模型通常足夠聰明，能識破基本拼寫錯誤並給出回應，但它也可能認為這是有意為之，並影響回應內容。

**檢查你的設定。** temperature 和 top_p 設定控制模型生成回應的決定性。如果你要模型給出只有一個正確答案的回應，則應將這些設定調低；如果你想要更多多樣化的回應，則可將它們調高。人們對這些設定最常犯的錯誤是誤以為它們是「聰明度」或「創意」的控制開關。


來源：https://learn.microsoft.com/azure/ai-foundry/openai/overview


### 5. 提交！


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

### 重覆執行相同的調用，結果有什麼比較？


In [ ]:

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

## 摘要文本  
#### 挑戰  
在文章結尾加入「tl;dr:」來摘要文本。注意模型如何在沒有額外指令的情況下理解並執行多項任務。你可以嘗試比 tl;dr 更具描述性的提示來調整模型行為並自訂所得到的摘要(3)。  

近期研究顯示，先在大型語料庫中進行預訓練，接著針對特定任務進行微調，可以在許多 NLP 任務和基準上大幅提升表現。雖然架構通常不依賴特定任務，但此方法仍然需要成千上萬筆特定任務的微調資料。相比之下，人類通常只需少量範例或簡單指示即可執行新的語言任務——這是當前 NLP 系統仍難以做到的事。本文展示了擴大語言模型規模能大幅提升不依賴特定任務的少量範例學習表現，有時甚至可與先前的微調頂尖方法相媲美。  



簡而言之  


# 多種使用案例練習  
1. 摘要文本  
2. 分類文本  
3. 產生新產品名稱


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## 分類文本  
#### 挑戰  
在推理時將項目分類到提供的類別中。在以下示例中，我們在提示中同時提供類別和要分類的文本(*playground_reference)。 

客戶查詢：您好，我的筆記本鍵盤上的一個鍵最近壞了，我需要更換：

分類類別：


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## 產生新產品名稱
#### 挑戰
從範例字詞中創造產品名稱。此處我們在提示中包含了關於我們將要為其創建名稱的產品資訊。我們也提供了一個相似的範例來展示我們期望接收的命名模式。我們還將溫度值設定得很高，以增加回應的隨機性及創新性。

產品描述：家用奶昔機
種子字詞：快速、健康、緊湊。
產品名稱：HomeShaker、Fit Shaker、QuickShake、Shake Maker

產品描述：一雙可適合任何腳型的鞋子。
種子字詞：可調適、合腳、全能合腳。


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}])

response.choices[0].message.content

# 參考資料  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [微調 GPT 模型以分類文本的最佳實踐](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# 尋求更多幫助  
[OpenAI Commercialization Team](AzureOpenAITeam@microsoft.com) 


# 貢獻者
* [Chew-Yean Yam](https://www.linkedin.com/in/cyyam/)


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們力求準確，但請注意，自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議尋求專業人工翻譯。我們不對因使用本翻譯而引起的任何誤解或曲解承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
